

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

In [26]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [27]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [28]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/


Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
Use "one shot prompting" in which we provide an example of how it should respond in the prompt.

In [31]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

Define a function to get the User Prompt

In [33]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [34]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

A call to gpt to select only the relevant links from the whole set of links identified in the prior step

In [35]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [36]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'resume', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'linkedin', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [37]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [38]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 6 relevant links


{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [39]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'GitHub page', 'url': 'https://github.com/huggingface'},
  {'type': 'LinkedIn page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'},
  {'type': 'Discord/community page',
   'url': 'https://huggingface.co/join/discord'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co/'},
  {'type': 'status page', 'url': 'https://status.huggingface.co/'},
  {'type': 'endpoints page', 'url': 'https://endpoints.huggingface.co'}]}



Assemble all the details into another prompt to GPT-5-nano to make the brochure!

In [40]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [41]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
baidu/Unlimited-OCR
Updated
4 days ago
•
630k
•
1.57k
empero-ai/Qwythos-9B-Claude-Mythos-5-1M-GGUF
Updated
3 days ago
•
1.11M
•
1.14k
zai-org/GLM-5.2
Updated
9 days ago
•
160k
•
3.16k
deepreinforce-ai/Ornith-1.0-35B-GGUF
Updated
6 days ago
•
234k
•
601
Qwen/Qwen-AgentWorld-35B-A3B
Updated
7 da

In [42]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [43]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [44]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nbaidu/Unlimited-OCR\nUpdated\n4 days ago\n•\n630k\n•\n1.57k\nempero-ai/Qwythos-9B-Claude-Mythos-5-1M-GGUF\nUpdated\n3 days ago

In [45]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [46]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 19 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is a pioneering AI company dedicated to building an open, collaborative, and ethical future for machine learning. Serving as the world’s largest AI community, it empowers machine learning engineers, researchers, scientists, and end-users to share, explore, and collaborate on advanced machine learning models, datasets, and applications.

At its core, Hugging Face is a vibrant platform driving the AI revolution by making state-of-the-art tools and resources accessible. It hosts over **2 million models** and **500,000 datasets**, providing a central hub for discovery, experimentation, and innovation in AI and machine learning.

---

## What Hugging Face Offers

- **Model Hub**: Browse and use a vast library of pre-trained open-source AI models across numerous tasks.
- **Datasets**: Access to a rich and growing collection of machine learning datasets to fuel training and research.
- **Spaces**: Deploy and interact with AI applications and demos, making AI accessible and engaging.
- **Enterprise Solutions**: Including Hugging Face PRO, enterprise support, inference endpoints, and storage solutions tailored for businesses.
- **Community Hubs**: Forums, Discord, GitHub, and blog posts enabling collaboration and knowledge-sharing.
- **Learning Resources**: Daily papers, tutorials, and documentation to empower users to stay ahead in AI.

---

## Company Culture

Hugging Face fosters a collaborative, open, and inclusive community culture that celebrates transparency and ethical AI development. The company thrives on sharing knowledge and building together, making AI technology accessible to everyone. Their fast-growing community includes thousands of contributors actively updating datasets, models, and scientific papers, which reflects a dynamic and innovative environment.

The culture encourages:

- **Open source collaboration** and community-driven development.
- Empowerment of the next generation of AI talent.
- Ethical innovation focused on building AI for good.
- Continuous learning and contribution through papers and discussions.

---

## Customers & Community

Hugging Face serves a diverse audience including:

- **Machine Learning Researchers** seeking open datasets and model implementations.
- **Data Scientists and Engineers** who require scalable frameworks and tools.
- **Enterprises** looking for AI deployment and inference solutions with enterprise-grade support.
- **AI Enthusiasts and Educators** accessing tutorials and community content.
- **Developers and Startups** building next-gen AI applications.

The platform is increasingly becoming the trusted ecosystem where AI applications are designed, trained, and deployed—accelerating innovation worldwide.

---

## Careers & Opportunities

Hugging Face is a rapidly growing company looking for passionate individuals eager to shape the future of AI. Career opportunities often focus on:

- Machine Learning Engineering
- Research Science
- Software Development
- Community and Developer Relations
- Product and Enterprise Solutions

Working at Hugging Face means joining a mission-driven team at the heart of cutting-edge AI technology, collaborating with a global community, and contributing to open-source projects that impact millions. 

Interested candidates can explore current openings and apply via the Hugging Face website.

---

## Connect with Hugging Face

- Explore Models and Datasets: https://huggingface.co/models
- Join the Community: Discord, Forum, GitHub
- Learn & Collaborate: Blog, Daily Papers, Documentation
- Follow on Social: Twitter | LinkedIn | GitHub

---

**Hugging Face** — *The AI community building the future.*  
Build, share, and innovate together with the world’s most vibrant machine learning platform.

---

**Brand Colors:**  
- Yellow: #FFD21E  
- Orange: #FF9D00  
- Gray: #6B7280  

**Official Logo and assets available for download on the website.**

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [47]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [48]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 8 relevant links


# Hugging Face Company Brochure

---

## About Hugging Face

**Hugging Face** is the AI community building the future of machine learning. It is a vibrant collaboration platform where individuals and organizations worldwide come together to create, discover, and advance machine learning models, datasets, and applications.

Hugging Face empowers the machine learning community by providing an open and collaborative environment featuring:

- Over 2 million machine learning models
- More than 500,000 datasets
- Thousands of machine learning applications and "Spaces"
- Tools for hosting and sharing public models and datasets without limits

It aims to make ML development and deployment faster, easier, and more accessible for researchers, developers, organizations, and enterprises alike.

---

## What Hugging Face Offers

### Community Platform  
A central hub for the AI and ML community to host, share, and explore models, datasets, and applications.

### Hugging Face Hub  
A collective repository with millions of models, datasets, and ML applications, enabling developers and researchers to collaborate seamlessly.

### HuggingChat  
An AI-powered chat service that showcases conversational AI capabilities.

### Spaces  
Interactive ML applications created by the community that anyone can explore or build upon.

### Enterprise Solutions  
- Hugging Face PRO for teams and enterprises  
- Enterprise support and deployment solutions  
- Inference providers and endpoints for scalable ML serving  
- Storage and efficient data bucket management

---

## Company Culture

Hugging Face fosters a **collaborative, open, and community-driven culture** that encourages knowledge sharing and innovation. The platform thrives on contributions from a broad, diverse community comprising researchers, developers, enterprises, and hobbyists, all working together to push the boundaries of AI and machine learning.

The company embraces openness, transparency, and a commitment to building the future of artificial intelligence in a way that benefits everyone.

---

## Customers and Community

- **Machine Learning Researchers & Developers:** Access to an unparalleled variety of models and datasets for research and development.
- **Enterprises:** Businesses leveraging Hugging Face’s enterprise offerings for AI model deployment, support, and scalable infrastructure.
- **Open Source Contributors:** Thousands of contributors who help expand and improve the platform weekly.
- **Educational Institutions and Students:** Utilizing the platform for learning and experimenting with AI technologies.
- **ML Application Creators:** Build and share interactive AI applications through Spaces.

---

## Careers at Hugging Face

Hugging Face offers exciting career opportunities for passionate individuals eager to contribute to AI innovation. The company looks for talents who are:

- Driven by a collaborative and open-source philosophy  
- Eager to advance state-of-the-art machine learning technologies  
- Skilled in AI research, software engineering, infrastructure, product development, and community building

Working at Hugging Face means joining a mission-driven team committed to shaping the future of AI with transparency and impact.

---

## Why Choose Hugging Face?

- **Scale & Diversity:** Access to the world’s largest and most up-to-date collection of models and datasets.
- **Community Focus:** Powered by an active and growing global community.
- **Innovation-Driven:** Always at the forefront of open AI research and production-ready ML tools.
- **Flexible Enterprise Support:** Robust offerings for teams and organizations requiring scalable, enterprise-grade solutions.
- **Open Source Spirit:** Strong commitment to open collaboration, ensuring transparency and democratization of AI technologies.

---

## Get Involved

- Explore AI Apps and browse over 2 million models and 500k datasets.  
- Join the community through Discord, Forum, GitHub, and Blogs.  
- Collaborate, share, or build your own ML applications in Hugging Face Spaces.  
- For enterprises, discover tailored solutions and professional support plans.

Visit [huggingface.co](https://huggingface.co) to learn more and join the future of AI.

---

## Brand Identity

- **Colors:** Signature colors include bright yellow (#FFD21E), warm orange (#FF9D00), and modern gray (#6B7280)  
- **Logo and Assets:** Available in vector and bitmap formats for use in projects aligned with Hugging Face branding  
- **Tagline:** "The AI community building the future."

---

Hugging Face is not just a platform—it’s a thriving, global AI community dedicated to promoting open innovation and driving the future of machine learning. Join now and be part of the revolution!

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links


# Hugging Face Brochure

---

## Who We Are  
Hugging Face is the AI community building the future. We serve as the collaboration platform for the global machine learning (ML) community, offering a centralized hub where ML engineers, scientists, and users can share, explore, discover, and experiment with open-source machine learning models, datasets, and applications. At the core of the AI revolution, Hugging Face empowers the next generation to build an open and ethical AI future, supported by a rapidly growing global community and innovative science teams.

---

## What We Offer  

### Our Platform  
- **Models:** Access and contribute to over 2 million machine learning models across a variety of tasks and domains.  
- **Datasets:** Browse and share from 500,000+ datasets curated for research, development, and production-ready solutions.  
- **Spaces:** Host and deploy ML applications and demos directly in the browser to showcase innovation and allow interactive demos.  
- **Buckets:** Scalable storage solutions designed for enterprise and team needs.  
- **HuggingChat:** Cutting-edge chatbot applications leveraging the latest in AI research and community models.  

### Enterprise Solutions  
Hugging Face supports organizations with dedicated team and enterprise plans that include:  
- Enterprise support and consultation  
- Inference endpoints and providers for production-grade AI deployment  
- Tailored solutions for team collaboration and large-scale ML management  
- Hugging Face PRO for advanced tooling and professional workflows  

---

## Our Culture  
- **Community-first:** We foster a vibrant and inclusive community where collaboration and open sharing are at the heart of progress.  
- **Open Source Commitment:** We believe in transparency and accessibility, offering some of the most widely used open-source ML tools and libraries globally.  
- **Innovation-driven:** Our talented science and engineering teams push the boundaries of AI research and practical applications.  
- **Ethical AI:** We emphasize building open and ethical AI, encouraging responsible development and deployment across industries.  

---

## Who Uses Hugging Face?  
Our platform is trusted by millions including:  
- Machine learning engineers and researchers looking for cutting-edge models and datasets  
- Developers building AI-powered applications and services  
- Enterprises deploying scalable, reliable AI infrastructure  
- Educational institutions and learners exploring AI technology  

---

## Careers at Hugging Face  
Join us in shaping the future of AI! We offer opportunities for talented professionals passionate about machine learning, software development, research, and community building. Working at Hugging Face means:  
- Collaborating with a fast-growing, world-class team  
- Contributing to open source projects impacting users worldwide  
- Engaging with a vibrant global AI community  
- Participating in an ethical and inclusive work environment  

Visit our [Careers page](https://huggingface.co/careers) to explore current openings.

---

## Connect With Us  
- Explore models, datasets, and spaces at [huggingface.co](https://huggingface.co)  
- Join the conversation on [Discord](https://discord.gg/huggingface), [GitHub](https://github.com/huggingface), [Twitter](https://twitter.com/huggingface), and [LinkedIn](https://www.linkedin.com/company/huggingface)  
- Learn and engage through our [Forum](https://discuss.huggingface.co) and [Blog](https://huggingface.co/blog)  

---

Hugging Face is more than a company — it’s a community dedicated to building the future of AI together.

**Hugging Face — The AI community building the future.**

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>